In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

ARTIFACTS_DIR = Path('../artifacts')
MODELS_DIR = ARTIFACTS_DIR / 'models'

### Let create dataframe for catboost training

In [2]:
num_float_features: int = 100
num_cat_features: int = 100
data_dim: int = 100_000

df = pd.DataFrame(dict(
    *[
        [(f'float_{i}', np.random.randn(data_dim)) for i in range(num_float_features)] + [(f'int_{i}', np.random.randint(-64, 64, data_dim)) for i in range(num_cat_features)]
    ],
))
df['target']  = (df.iloc[:, :num_float_features].pow(1).mean(1) * df.iloc[:, num_float_features:num_float_features+num_cat_features].pow(1).mean(1)) > 0
float_features_names = [f'float_{i}' for i in range(num_float_features)]
cat_features_names = [f'int_{i}' for i in range(num_cat_features)]
all_features = float_features_names + cat_features_names
df

,float_0,float_1,float_2,float_3,float_4,float_5,float_6,float_7,float_8,float_9,...,int_91,int_92,int_93,int_94,int_95,int_96,int_97,int_98,int_99,target
0,1.423607,-1.858332,-1.194514,1.072676,1.386371,0.420614,1.148491,-0.444261,0.388134,0.392965,...,-27,-45,47,-62,6,47,-9,-57,16,True
1,0.041759,1.056737,-0.352343,-0.248840,0.471696,1.994127,-0.600270,1.295830,1.195552,0.789697,...,31,-41,57,-40,-47,19,-57,3,53,False
2,0.088881,-0.780143,-1.309660,-0.711827,1.061731,-1.399934,0.318256,-0.650679,-1.083158,-0.473470,...,-60,-50,-43,51,19,-28,18,-49,1,True
3,0.833975,-0.644847,-2.324448,0.657893,0.795349,-0.163799,0.455593,-0.683077,0.062645,1.338297,...,-23,9,-4,-18,-1,-52,31,-13,8,True
4,1.184944,1.174786,1.451164,-1.652390,-0.277764,-0.273603,-0.490021,1.652827,-2.381516,1.722875,...,28,-45,-23,-43,-11,-14,-30,-17,44,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,-0.587553,1.159750,1.829377,1.439462,-0.865440,1.543928,-1.423441,-0.223704,0.125801,1.561161,...,-62,30,-63,10,-21,60,-47,-60,-17,False
99996,1.357612,2.139648,0.013729,0.963130,0.399363,-0.302323,0.186820,0.368358,0.551692,-0.130634,...,34,-30,-8,-16,25,-23,20,-27,-24,True
99997,-0.419170,0.345059,0.820997,0.615086,-0.197736,0.197611,0.005006,-0.586986,1.786309,0.403754,...,51,28,6,-60,-56,59,11,-2,55,False
99998,0.186774,0.035699,0.102338,0.529391,-0.231834,-0.171270,1.649694,0.543455,1.220052,-0.697168,...,-43,24,9,-49,-24,-21,41,-35,-1,False


### Model training

In [3]:
from catboost import CatBoostClassifier, FeaturesData, Pool
from sklearn.metrics import accuracy_score

In [4]:
train_pool = Pool(
    data=df[all_features],
    cat_features=cat_features_names,
    label=df['target'],
)

model = CatBoostClassifier(iterations=200)
model.fit(train_pool)
model.save_model(MODELS_DIR/'cb_big_cls.json', format='json')
model.save_model(MODELS_DIR/'cb_big_cls.cpp', format='cpp', pool=train_pool)

Learning rate set to 0.322025
0:	learn: 0.6928816	total: 216ms	remaining: 42.9s
1:	learn: 0.6924663	total: 361ms	remaining: 35.7s
2:	learn: 0.6921632	total: 500ms	remaining: 32.9s
3:	learn: 0.6917962	total: 646ms	remaining: 31.6s
4:	learn: 0.6915111	total: 786ms	remaining: 30.7s
5:	learn: 0.6912222	total: 902ms	remaining: 29.2s
6:	learn: 0.6908100	total: 999ms	remaining: 27.6s
7:	learn: 0.6904973	total: 1.12s	remaining: 26.9s
8:	learn: 0.6901384	total: 1.24s	remaining: 26.4s
9:	learn: 0.6897739	total: 1.36s	remaining: 25.8s
10:	learn: 0.6895553	total: 1.48s	remaining: 25.4s
11:	learn: 0.6892260	total: 1.6s	remaining: 25.1s
12:	learn: 0.6888577	total: 1.74s	remaining: 25s
13:	learn: 0.6885925	total: 1.89s	remaining: 25.1s
14:	learn: 0.6882030	total: 2.02s	remaining: 24.9s
15:	learn: 0.6878440	total: 2.16s	remaining: 24.8s
16:	learn: 0.6875308	total: 2.29s	remaining: 24.7s
17:	learn: 0.6872640	total: 2.41s	remaining: 24.4s
18:	learn: 0.6869968	total: 2.57s	remaining: 24.5s
19:	learn: 0.6

In [2]:
from cats_compiler.compiler.ast.parser import parse_model_file_from_json
model_file = parse_model_file_from_json(MODELS_DIR/'cb_classifier.json')

In [6]:
model_file.trees

In [6]:
model_file.model_file.keys()

dict_keys(['features_info', 'model_info', 'oblivious_trees', 'scale_and_bias'])

In [ ]:
model_file.model_file['features_info'] # ['features_info'] # ['float_features']

{'float_features': [{'borders': [-0.9822249412536621,
    -0.7023289203643799,
    -0.357035756111145,
    -0.23571404814720154,
    -0.06725707650184631,
    0.5741671323776245],
   'feature_id': 'float_0',
   'feature_index': 0,
   'flat_feature_index': 0,
   'has_nans': False,
   'nan_value_treatment': 'AsIs'},
  {'borders': [-0.8006176948547363,
    -0.7057121396064758,
    -0.6620782613754272,
    -0.46881240606307983,
    -0.07174383103847504,
    0.15795378386974335,
    0.41341835260391235,
    0.8021426200866699],
   'feature_id': 'float_1',
   'feature_index': 1,
   'flat_feature_index': 1,
   'has_nans': False,
   'nan_value_treatment': 'AsIs'},
  {'borders': [2.5, 3.5, 4.5],
   'feature_id': 'int_0',
   'feature_index': 2,
   'flat_feature_index': 2,
   'has_nans': False,
   'nan_value_treatment': 'AsIs'},
  {'borders': [1.5, 2.5, 3.5, 4.5, 5.5],
   'feature_id': 'int_1',
   'feature_index': 3,
   'flat_feature_index': 3,
   'has_nans': False,
   'nan_value_treatment': 'AsI

In [7]:
model_file.model_file['features_info']['float_features'][0]

{'borders': [-0.9822249412536621,
  -0.7023289203643799,
  -0.357035756111145,
  -0.23571404814720154,
  -0.06725707650184631,
  0.5741671323776245],
 'feature_id': 'float_0',
 'feature_index': 0,
 'flat_feature_index': 0,
 'has_nans': False,
 'nan_value_treatment': 'AsIs'}

In [18]:
model_file.model_file['oblivious_trees'][0]['splits']

[{'border': -0.23571404814720154,
  'float_feature_index': 0,
  'split_index': 3,
  'split_type': 'FloatFeature'},
 {'border': -0.06725707650184631,
  'float_feature_index': 0,
  'split_index': 4,
  'split_type': 'FloatFeature'},
 {'border': 3.5,
  'float_feature_index': 2,
  'split_index': 15,
  'split_type': 'FloatFeature'}]